<a href="https://colab.research.google.com/github/Danzigerrr/MultiClass-Entity-Linking-System/blob/NER-datasets/NED_simple_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import json

In [6]:

# Define the classes to structure the data
class TestEntity:
    def __init__(self, surface_form, ner_class, position, target_uri):
        self.surface_form = surface_form
        self.ner_class = ner_class
        self.position = tuple(position)  # Convert to tuple
        self.target_uri = target_uri

    def __repr__(self):
        return f"TestEntity(surface_form={self.surface_form}, ner_class={self.ner_class}, position={self.position}, target_uri={self.target_uri})"

class TestText:
    def __init__(self, text, entity_mentions):
        self.text = text
        self.entity_mentions = [TestEntity(**entity) for entity in entity_mentions]

    def __repr__(self):
        return f"TestText(text={self.text}, entity_mentions={self.entity_mentions})"


In [23]:

# Step 1: Fetch the data from the URL
url = "https://raw.githubusercontent.com/Entity-Linking-Smart-Boys/EntityLinkingSystem/refs/heads/main/test_datasets/ace2004_test.json"
response = requests.get(url)

# Step 2: Check if the request was successful
if response.status_code == 200:
    # Step 3: Load the JSON data into Python
    raw_data = response.json()

    print(raw_data)
else:
    print(f"Failed to retrieve data. HTTP Status code: {response.status_code}")

In [37]:
import json
import requests
from typing import List

class Position:
    def __init__(self, start: int, end: int):
        self.start = start
        self.end = end

    def __repr__(self):
        return f"Position({self.start}, {self.end})"

    def to_dict(self):
        # Convert Position object to a dictionary
        return {'start': self.start, 'end': self.end}

class Entity:
    def __init__(self, surface_form: str, position: Position, dbpedia_target_uri: str):
        self.surface_form = surface_form
        self.position = position
        self.dbpedia_target_uri = dbpedia_target_uri
        self.wikidata_target_uri = self.get_wikidata_target_uri()

    def __repr__(self):
        return f"Entity(surface_form={self.surface_form}, position={self.position}, dbpedia_target_uri={self.dbpedia_target_uri}, wikidata_target_uri={self.wikidata_target_uri})"

    def print_entity_details(self):
        print(f"Entity Details:")
        print(f"  Surface Form: {self.surface_form}")
        print(f"  Position: {self.position.start} to {self.position.end}")
        print(f"  DBpedia Target URI: {self.dbpedia_target_uri}")
        print(f"  WikiData Target URI: {self.wikidata_target_uri}")
        print("-" * 40)

    def get_wikidata_target_uri(self):
        # Extract the label from DBpedia URI (last part of the URI)
        label = self.dbpedia_target_uri.split("/")[-1]

        # Query Wikidata API to find the Wikidata URI for the DBpedia label
        wikidata_url = f"https://www.wikidata.org/w/api.php?action=wbsearchentities&search={label}&language=en&format=json"

        try:
            response = requests.get(wikidata_url)
            data = response.json()

            # Check if any results were returned
            if 'search' in data and data['search']:
                # Use the first search result as the most relevant one
                wikidata_id = data['search'][0]['id']
                return f"https://www.wikidata.org/wiki/{wikidata_id}"
            else:
                return "Wikidata URI not found"
        except Exception as e:
            return f"Error fetching Wikidata URI: {str(e)}"

    def to_dict(self):
        # Convert Entity object to a dictionary
        return {
            'surface_form': self.surface_form,
            'position': self.position.to_dict(),
            'dbpedia_target_uri': self.dbpedia_target_uri,
            'wikidata_target_uri': self.wikidata_target_uri
        }

class Text:
    def __init__(self, text: str, entity_mentions: List[Entity]):
        self.text = text
        self.entity_mentions = entity_mentions

    def __repr__(self):
        return f"Text(text={self.text[:50]}..., entity_mentions={self.entity_mentions})"  # Preview the text

    def print_entity_details(self):
        print(f"Text: {self.text[:100]}...")  # Show a preview of the text
        print("=" * 40)
        for entity in self.entity_mentions:
            entity.print_entity_details()

    def to_dict(self):
        # Convert Text object to a dictionary
        return {
            'text': self.text,
            'entity_mentions': [entity.to_dict() for entity in self.entity_mentions]
        }

def convert_data(data: str) -> List[Text]:
    parsed_data = json.loads(data)

    texts = []
    for item in parsed_data:
        text = item['text']
        entity_mentions = []

        for entity in item['entity_mentions']:
            surface_form = entity['surface_form']
            position = Position(*entity['position']['py/tuple'])
            target_uri = entity['target_uri']

            entity_obj = Entity(surface_form, position, target_uri)
            entity_mentions.append(entity_obj)

        text_obj = Text(text, entity_mentions)
        texts.append(text_obj)

    return texts

def save_to_json(file_path: str, data: List[Text]):
    # Convert the list of Text objects to a JSON-serializable format
    dict_data = [text.to_dict() for text in data]

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(dict_data, f, ensure_ascii=False, indent=4)

# Convert data into structured information
structured_data = convert_data(raw_data)

# Print entity details in a structured way
for text in structured_data[:1]:
    text.print_entity_details()


In [38]:
# Save structured data to a JSON file
save_to_json('structured_data.json', structured_data)

